# Proyecto 5: Segmentación de Clientes mediante Clustering
## Notebook 6: DBSCAN (Density-Based Spatial Clustering)

### Objetivo
Implementar DBSCAN como método de clustering basado en densidad para identificar outliers y clusters de forma irregular.

### DBSCAN: Características Principales
**Definición:** Agrupa puntos que están densamente conectados, identificando también puntos de ruido/outliers.

**Parámetros Clave:**
- **eps (ε)**: Radio de vecindad máxima
- **min_samples**: Número mínimo de puntos en ε-vecindad para core point

**Ventajas:**
- Descubre clusters de forma irregular
- Identifica outliers automáticamente
- No requiere especificar número de clusters
- Basado en densidad (más robusto)

**Desventajas:**
- Sensible a parámetros eps y min_samples
- Dificultad con datos de densidad variable
- Requiere escalado apropiado de datos

## 1. Importación y Preparación

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score
import warnings
warnings.filterwarnings('ignore')

# Configuración visual
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("✓ Librerías importadas")

In [ ]:
# Cargar y preparar datos
ruta_datos = '../data/Wholesale customers data.csv'
df_original = pd.read_csv(ruta_datos)

variables_numericas = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
scaler = StandardScaler()
df_estandarizado = df_original.copy()
df_estandarizado[variables_numericas] = scaler.fit_transform(df_original[variables_numericas])

X = df_estandarizado[variables_numericas].values

print(f"✓ Datos cargados: {X.shape}")

## 2. Selección del Parámetro eps usando k-distance Graph

In [ ]:
# Calcular k-distance graph para determinar eps
# Usar min_samples = n_features * 2 como sugerencia inicial
k = 6  # número de dimensiones
neighbors = NearestNeighbors(n_neighbors=k)
neighbors_fit = neighbors.fit(X)
distances, indices = neighbors_fit.kneighbors(X)

# Ordenar distancias
distances = np.sort(distances[:, k-1], axis=0)

# Visualizar k-distance graph
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico completo
axes[0].plot(distances, linewidth=1)
axes[0].set_xlabel('Índice de Puntos (ordenados)', fontweight='bold')
axes[0].set_ylabel(f'Distancia al {k}-ésimo Vecino Más Cercano', fontweight='bold')
axes[0].set_title('k-distance Graph (Completo)', fontsize=11, fontweight='bold')
axes[0].grid(alpha=0.3)
axes[0].axhline(y=0.5, color='red', linestyle='--', label='eps=0.5 sugerido')
axes[0].axhline(y=0.6, color='orange', linestyle='--', label='eps=0.6')
axes[0].legend()

# Gráfico zoom en codo
axes[1].plot(distances[-100:], linewidth=1.5, marker='o', markersize=3)
axes[1].set_xlabel('Índice de Puntos (últimos 100)', fontweight='bold')
axes[1].set_ylabel(f'Distancia al {k}-ésimo Vecino', fontweight='bold')
axes[1].set_title('k-distance Graph (Últimos 100 puntos)', fontsize=11, fontweight='bold')
axes[1].grid(alpha=0.3)
axes[1].axhline(y=0.5, color='red', linestyle='--', label='eps=0.5')
axes[1].axhline(y=0.6, color='orange', linestyle='--', label='eps=0.6')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"✓ k-distance graph calculado")
print(f"Recomendación visual: eps ≈ 0.5-0.7")

## 3. DBSCAN con eps=0.5, min_samples=5

In [ ]:
# Aplicar DBSCAN
eps_param = 0.5
min_samples_param = 5

dbscan = DBSCAN(eps=eps_param, min_samples=min_samples_param)
labels_dbscan = dbscan.fit_predict(X)

# Agregar al dataframe
df_estandarizado['DBSCAN'] = labels_dbscan
df_original['DBSCAN'] = labels_dbscan

# Numero de clusters
n_clusters = len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
n_outliers = list(labels_dbscan).count(-1)

print(f"✓ DBSCAN aplicado (eps={eps_param}, min_samples={min_samples_param})")
print(f"\nResultados:")
print(f"  • Número de clusters: {n_clusters}")
print(f"  • Número de outliers (ruido): {n_outliers} ({n_outliers/len(X)*100:.1f}%)")
print(f"\nDistribución de clusters:")

for label in sorted(set(labels_dbscan)):
    count = list(labels_dbscan).count(label)
    if label == -1:
        print(f"  • Outliers: {count} puntos")
    else:
        print(f"  • Cluster {label}: {count} puntos ({count/len(X)*100:.1f}%)")

## 4. Visualización de Clusters en PCA

In [ ]:
# Aplicar PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# Visualizar clusters
plt.figure(figsize=(12, 8))

# Colores para clusters (excluir outliers inicialmente)
colores_cluster = sns.color_palette('husl', n_clusters)

# Plotear clusters
for i in range(n_clusters):
    mask = labels_dbscan == i
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], 
               c=[colores_cluster[i]], label=f'Cluster {i}',
               s=70, alpha=0.7, edgecolors='black', linewidth=0.5)

# Plotear outliers en diferente color
if n_outliers > 0:
    mask_outliers = labels_dbscan == -1
    plt.scatter(X_pca[mask_outliers, 0], X_pca[mask_outliers, 1],
               c='red', marker='X', s=150, alpha=0.8, edgecolors='darkred', linewidth=1.5,
               label=f'Outliers (n={n_outliers})')

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontweight='bold', fontsize=11)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontweight='bold', fontsize=11)
plt.title(f'DBSCAN Clustering (eps={eps_param}, min_samples={min_samples_param})\nVisualización en 2D usando PCA',
         fontsize=13, fontweight='bold', pad=15)
plt.legend(loc='best', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✓ Visualización en PCA completada")

## 5. Análisis de Outliers

In [ ]:
print("\nANÁLISIS DE OUTLIERS DETECTADOS\n" + "="*70)

if n_outliers > 0:
    outliers_df = df_original[df_original['DBSCAN'] == -1]
    
    print(f"\nTotal de outliers: {n_outliers}")
    print(f"Porcentaje del dataset: {n_outliers/len(df_original)*100:.1f}%")
    
    print(f"\nCaracterísticas promedio de outliers:")
    for var in variables_numericas:
        print(f"  • {var:20s}: ${outliers_df[var].mean():>10,.0f}")
    
    print(f"\nCanales en outliers:")
    print(outliers_df['Channel'].value_counts())
    
    print(f"\nRegiones en outliers:")
    print(outliers_df['Region'].value_counts())
else:
    print("No se detectaron outliers con los parámetros actuales")

print("\nANÁLISIS DE CLUSTERS ENCONTRADOS (DBSCAN)\n" + "="*70)

for cluster in range(n_clusters):
    df_cluster = df_original[df_original['DBSCAN'] == cluster]
    
    print(f"\nCluster {cluster} (n={len(df_cluster)} | {len(df_cluster)/len(df_original)*100:.1f}%):")
    print("-" * 70)
    
    # Gasto promedio
    print("Gasto promedio:")
    for var in variables_numericas:
        print(f"  • {var:20s}: ${df_cluster[var].mean():>10,.0f}")
    
    # Composición
    print(f"\nCanal predominante: {df_cluster['Channel'].mode()[0]} ({df_cluster['Channel'].value_counts().iloc[0]} clientes)")

## 7. Evaluación de Diferentes Parámetros eps

In [ ]:
# Evaluar diferentes valores de eps
eps_values = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
resultados_dbscan = {}

print("Evaluando diferentes valores de eps...\n")
print(f"{'eps':<6} {'Clusters':<12} {'Outliers':<12} {'Silhouette':<15}")
print("-" * 50)

for eps in eps_values:
    dbscan_temp = DBSCAN(eps=eps, min_samples=5)
    labels_temp = dbscan_temp.fit_predict(X)
    
    n_clusters_temp = len(set(labels_temp)) - (1 if -1 in labels_temp else 0)
    n_outliers_temp = list(labels_temp).count(-1)
    
    # Silhouette solo si hay clusters válidos
    if n_clusters_temp > 1 and n_outliers_temp < len(X) - 1:
        try:
            silhouette = silhouette_score(X[labels_temp != -1], labels_temp[labels_temp != -1])
        except:
            silhouette = np.nan
    else:
        silhouette = np.nan
    
    resultados_dbscan[eps] = {
        'n_clusters': n_clusters_temp,
        'n_outliers': n_outliers_temp,
        'silhouette': silhouette,
        'labels': labels_temp
    }
    
    print(f"{eps:<6.1f} {n_clusters_temp:<12} {n_outliers_temp:<12} {silhouette:<15.3f}")

In [ ]:
# Visualizar resultados por eps
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

eps_list = sorted(resultados_dbscan.keys())
clusters_count = [resultados_dbscan[eps]['n_clusters'] for eps in eps_list]
outliers_count = [resultados_dbscan[eps]['n_outliers'] for eps in eps_list]
silhouettes = [resultados_dbscan[eps]['silhouette'] if not np.isnan(resultados_dbscan[eps]['silhouette']) else 0 for eps in eps_list]

# Número de clusters
axes[0].plot(eps_list, clusters_count, 'o-', linewidth=2, markersize=8, color='steelblue')
axes[0].axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='Actual (eps=0.5)')
axes[0].set_xlabel('eps', fontweight='bold')
axes[0].set_ylabel('Número de Clusters', fontweight='bold')
axes[0].set_title('Clusters vs eps', fontsize=11, fontweight='bold')
axes[0].grid(alpha=0.3)
axes[0].legend()

# Número de outliers
axes[1].plot(eps_list, outliers_count, 'o-', linewidth=2, markersize=8, color='coral')
axes[1].axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='Actual (eps=0.5)')
axes[1].set_xlabel('eps', fontweight='bold')
axes[1].set_ylabel('Número de Outliers', fontweight='bold')
axes[1].set_title('Outliers vs eps', fontsize=11, fontweight='bold')
axes[1].grid(alpha=0.3)
axes[1].legend()

# Silhouette score
axes[2].plot(eps_list, silhouettes, 'o-', linewidth=2, markersize=8, color='green')
axes[2].axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='Actual (eps=0.5)')
axes[2].set_xlabel('eps', fontweight='bold')
axes[2].set_ylabel('Silhouette Score', fontweight='bold')
axes[2].set_title('Silhouette vs eps', fontsize=11, fontweight='bold')
axes[2].grid(alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()

## 8. Análisis de Distancias Intra-cluster

## 9. Comparación de Métodos: K-means vs Jerárquico vs DBSCAN

In [ ]:
print("\nCOMPARACIÓN DE LOS TRES MÉTODOS DE CLUSTERING\n" + "="*80)

# Obtener labels de K-means
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_kmeans = kmeans.fit_predict(X)

# Obtener labels de Jerárquico
from scipy.cluster.hierarchy import linkage, fcluster
Z_ward = linkage(X, method='ward')
labels_jerarquico = fcluster(Z_ward, 3, criterion='maxclust') - 1

# Tabla comparativa
metodos = ['K-means', 'Jerárquico', 'DBSCAN']
n_clusters_comp = [3, 3, n_clusters]
n_outliers_comp = [0, 0, n_outliers]

# Calcular métricas (solo para puntos no-outlier en DBSCAN)
labels_dbscan_valid = labels_dbscan[labels_dbscan != -1]
X_valid = X[labels_dbscan != -1]

silhouette_kmeans = silhouette_score(X, labels_kmeans)
silhouette_jerarquico = silhouette_score(X, labels_jerarquico)
if len(X_valid) > 0:
    silhouette_dbscan = silhouette_score(X_valid, labels_dbscan_valid)
else:
    silhouette_dbscan = np.nan

comparacion_df = pd.DataFrame({
    'Método': metodos,
    'Clusters': n_clusters_comp,
    'Outliers': n_outliers_comp,
    'Silhouette': [silhouette_kmeans, silhouette_jerarquico, silhouette_dbscan]
})

print("\n" + comparacion_df.to_string(index=False))
print("\n" + "="*80)

## 10. Visualización 3D

## 11. Resumen y Conclusiones

In [ ]:
resumen = f"""
{'='*80}
RESUMEN - DBSCAN CLUSTERING
{'='*80}

1. PARÁMETROS UTILIZADOS:
   • eps (ε): {eps_param}
   • min_samples: {min_samples_param}
   • Métrica de distancia: Euclidiana

2. RESULTADOS:
   • Número de clusters encontrados: {n_clusters}
   • Número de outliers detectados: {n_outliers} ({n_outliers/len(X)*100:.1f}%)
   • Silhouette Score (sin outliers): {silhouette_dbscan:.3f if not np.isnan(silhouette_dbscan) else 'N/A'}

3. CARACTERÍSTICAS DEL MÉTODO:
   ✓ Descubrimiento automático de número de clusters
   ✓ Identificación de puntos de ruido/outliers
   ✓ No requiere especificar k previamente
   ✓ Basado en densidad (diferente a K-means)

4. VENTAJAS DETECTADAS:
   ✓ Identifica {n_outliers} outliers potenciales para análisis especial
   ✓ Descubre clusters de forma irregular
   ✓ Robusto para datos de densidad variable

5. LIMITACIONES:
   ⚠ Sensible a selección de eps
   ⚠ Requiere ajuste empírico de parámetros
   ⚠ Dificultad con clusters de densidad muy diferente

6. COMPARACIÓN GLOBAL (3 MÉTODOS):
   • K-means: Mejor para clusters esféricos, rápido
   • Jerárquico: Dendrogram informativo, también da k=3
   • DBSCAN: Identifica outliers, clusters irregulares

7. RECOMENDACIÓN PARA NEGOCIO:
   → K-means o Jerárquico para segmentación estándar (k=3)
   → DBSCAN para identificación de outliers/casos especiales
   → Los {n_outliers} outliers merecen análisis especial

{'='*80}
"""

print(resumen)

In [ ]:
print("✓ Análisis DBSCAN completado")
print("✓ Todos los tres métodos evaluados y comparados")